<a href="https://colab.research.google.com/github/stekkos89thelawnmower/Flamingo-Qd-Analysis/blob/main/FLAMINGO_QD_mass_dependent_signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FLAMINGO Q_D — Halo-Mass-Dependent NoCooling Signal

Self-contained pipeline consolidating the full mass-binned differential analysis at z=0.0: mass-bin profile detection, specificity screening (Jet, fgas-8sigma) with multiple-comparisons correction, and a four-level robustness protocol (standard paired test, spatial jackknife, multi-seed replication, fixed-grid cross-check) applied to both the low-mass and high-mass signal.

Companion report: *Halo-Mass-Dependent NoCooling Differential Signal in FLAMINGO*, S. Boi (2026).

## 0. Setup

In [ ]:
!pip install hdfstream -q
import numpy as np
import hdfstream
import time
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import wilcoxon


In [ ]:
BOX_SIDE = 1000.0
GRAMS_PER_MSUN = 1.988409870698051e33
CM_PER_KM = 1e5
MASS_CUT = 1e12
N_GRID = 200
CELL_SIZE = BOX_SIDE / N_GRID
K_NEIGHBORS = 16
GRID_1D = np.linspace(0, BOX_SIDE, N_GRID, endpoint=False) + CELL_SIZE / 2
SEED = 42

MASS_LO_EDGE = 1.21e12   # fixed low-mass bin edge, validated robust
MASS_HI_EDGE = 5.91e12   # fixed high-mass bin edge, validated robust
N_TRANSITION_BINS = 3    # widened transition zone, 3 equipopulated bins

SNAPSHOT_Z0 = "halo_properties_0077.hdf5"  # z=0.0


In [ ]:
def load_tracers_velocities_masses(flamingo_dir, run_path, snapshot_file, expected_z=None):
    """Load halo (M > MASS_CUT) positions, peculiar velocities, and masses."""
    soap_file = flamingo_dir[f"{run_path}/SOAP-HBT/{snapshot_file}"]
    z = float(np.array(soap_file["Header"].attrs["Redshift"]).squeeze())
    if expected_z is not None:
        assert abs(z - expected_z) < 1e-6, f"unexpected z: {z} (expected {expected_z})"
    total_mass_raw = np.array(soap_file["SO/200_crit/TotalMass"][:], dtype=np.float64)
    conv_mass = dict(soap_file["SO/200_crit/TotalMass"].attrs)
    cgs_mass = float(np.array(conv_mass["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    mass_msun = total_mass_raw * cgs_mass / GRAMS_PER_MSUN
    pos_raw = np.mod(np.array(soap_file["SO/200_crit/CentreOfMass"][:]), BOX_SIDE)
    vel_raw = np.array(soap_file["SO/200_crit/CentreOfMassVelocity"][:], dtype=np.float64)
    conv_vel = dict(soap_file["SO/200_crit/CentreOfMassVelocity"].attrs)
    vel_cgs_factor = float(np.array(conv_vel["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    vel_kms = vel_raw * vel_cgs_factor / CM_PER_KM
    mask = mass_msun > MASS_CUT
    return pos_raw[mask], vel_kms[mask], mass_msun[mask], z


def periodic_delta(a, b, box_side):
    d = np.abs(a - b)
    return np.minimum(d, box_side - d)


def make_observer_positions(n_observers, min_separation_mpc, box_side, seed=SEED, max_tries_factor=500):
    rng = np.random.default_rng(seed)
    positions = []
    tries = 0
    max_tries = max_tries_factor * n_observers
    while len(positions) < n_observers and tries < max_tries:
        cand = rng.uniform(0, box_side, size=3)
        if all(np.sqrt((periodic_delta(cand, p, box_side) ** 2).sum()) >= min_separation_mpc for p in positions):
            positions.append(cand)
        tries += 1
    return np.array(positions)


def make_grid_observer_positions(n_per_axis, box_side):
    """Regular Cartesian grid, n_per_axis^3 points -- deterministic cross-check."""
    spacing = box_side / n_per_axis
    coords_1d = np.arange(n_per_axis) * spacing + spacing / 2
    gx, gy, gz = np.meshgrid(coords_1d, coords_1d, coords_1d, indexing="ij")
    return np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)


def build_grid_tree(grid_1d, box_side):
    gx, gy, gz = np.meshgrid(grid_1d, grid_1d, grid_1d, indexing="ij")
    grid_points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    return cKDTree(grid_points, boxsize=box_side)


def subsample_to_match_with_mass(tracers, velocities, masses, target_n, seed):
    n = len(tracers)
    if n <= target_n:
        return tracers, velocities, masses
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=target_n, replace=False)
    return tracers[idx], velocities[idx], masses[idx]


In [ ]:
VARIANT_A = {"run_path": "L1_m9/L1_m9", "label": "fiducial"}
VARIANT_B = {"run_path": "L1_m9/NoCooling", "label": "NoCooling"}
VARIANT_JET = {"run_path": "L1_m9/Jet", "label": "Jet"}
VARIANT_FGAS8SIGMA = {"run_path": "L1_m9/fgas-8sigma", "label": "fgas-8sigma"}
COMPARISON_VARIANTS = [VARIANT_B, VARIANT_JET, VARIANT_FGAS8SIGMA]

root_dir = hdfstream.open("cosma", "/")
flamingo_dir = root_dir["FLAMINGO"]

grid_tree = build_grid_tree(GRID_1D, BOX_SIDE)


## 1. Baseline Q_D engine and statistical toolkit

IDW velocity field (unchanged, validated engine), the standard paired bootstrap/Wilcoxon comparison, and the spatial block jackknife (octant and finer 27-block) used throughout the robustness protocol.

In [ ]:
def build_velocity_field(tracers, velocities, grid_1d, box_side, n_grid, k_neighbors):
    tree = cKDTree(tracers, boxsize=box_side)
    vx = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    vy = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    vz = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    for i in range(n_grid):
        gx = np.full(n_grid * n_grid, grid_1d[i])
        gyv, gzv = np.meshgrid(grid_1d, grid_1d, indexing="ij")
        pts = np.stack([gx, gyv.ravel(), gzv.ravel()], axis=1)
        d, idx = tree.query(pts, k=k_neighbors)
        d = np.maximum(d, 1e-6)
        w = 1.0 / d**2
        w /= w.sum(axis=1, keepdims=True)
        vx[i] = (w * velocities[idx, 0]).sum(axis=1).reshape(n_grid, n_grid)
        vy[i] = (w * velocities[idx, 1]).sum(axis=1).reshape(n_grid, n_grid)
        vz[i] = (w * velocities[idx, 2]).sum(axis=1).reshape(n_grid, n_grid)
    return vx, vy, vz


def compute_theta_sigma2(vx, vy, vz, cell_size):
    dvx_dx = (np.roll(vx,-1,0)-np.roll(vx,1,0))/(2*cell_size)
    dvx_dy = (np.roll(vx,-1,1)-np.roll(vx,1,1))/(2*cell_size)
    dvx_dz = (np.roll(vx,-1,2)-np.roll(vx,1,2))/(2*cell_size)
    dvy_dx = (np.roll(vy,-1,0)-np.roll(vy,1,0))/(2*cell_size)
    dvy_dy = (np.roll(vy,-1,1)-np.roll(vy,1,1))/(2*cell_size)
    dvy_dz = (np.roll(vy,-1,2)-np.roll(vy,1,2))/(2*cell_size)
    dvz_dx = (np.roll(vz,-1,0)-np.roll(vz,1,0))/(2*cell_size)
    dvz_dy = (np.roll(vz,-1,1)-np.roll(vz,1,1))/(2*cell_size)
    dvz_dz = (np.roll(vz,-1,2)-np.roll(vz,1,2))/(2*cell_size)
    theta = dvx_dx+dvy_dy+dvz_dz
    grad = np.array([[dvx_dx,dvx_dy,dvx_dz],[dvy_dx,dvy_dy,dvy_dz],[dvz_dx,dvz_dy,dvz_dz]])
    sym = 0.5*(grad+grad.transpose(1,0,2,3,4))
    trace_third = theta/3.0
    sigma = sym.copy()
    for i in range(3):
        sigma[i,i] -= trace_third
    sigma2 = np.sum(sigma**2, axis=(0,1))
    return theta, sigma2


def bootstrap_ci(delta, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(delta)
    boot_means = np.array([rng.choice(delta, size=n, replace=True).mean() for _ in range(n_boot)])
    return np.percentile(boot_means, [2.5, 97.5])


def QD_per_domain(theta, sigma2, grid_tree, observer_positions, radius_mpc):
    theta_flat = theta.ravel(); sigma2_flat = sigma2.ravel()
    qd_values = np.empty(len(observer_positions))
    for i, obs in enumerate(observer_positions):
        idx = grid_tree.query_ball_point(obs, r=radius_mpc)
        th = theta_flat[idx]; s2 = sigma2_flat[idx]
        var_theta = (th**2).mean() - th.mean()**2
        qd_values[i] = (2.0/3.0)*var_theta - s2.mean()
    return qd_values


def paired_QD_comparison(theta_A, sigma2_A, theta_B, sigma2_B, grid_tree,
                          observer_positions, radius_mpc, label_A="A", label_B="B", verbose=True):
    """delta = qd_B - qd_A (A=fiducial, B=comparison variant)."""
    qd_A = QD_per_domain(theta_A, sigma2_A, grid_tree, observer_positions, radius_mpc)
    qd_B = QD_per_domain(theta_B, sigma2_B, grid_tree, observer_positions, radius_mpc)
    delta = qd_B - qd_A
    ci_lo, ci_hi = bootstrap_ci(delta)
    try:
        _, p_wilcoxon = wilcoxon(qd_B, qd_A)
    except ValueError:
        p_wilcoxon = float("nan")
    if verbose:
        print(f"R={radius_mpc:.0f} Mpc, N={len(observer_positions)}: "
              f"{label_A}={qd_A.mean():.3f}  {label_B}={qd_B.mean():.3f}  "
              f"Delta={delta.mean():.3f}  CI95%=[{ci_lo:.3f},{ci_hi:.3f}]  p_Wilcoxon={p_wilcoxon:.2e}")
    return {"qd_A": qd_A, "qd_B": qd_B, "delta": delta, "ci_lo": ci_lo, "ci_hi": ci_hi, "p_wilcoxon": p_wilcoxon}


def assign_octant(positions, box_side):
    half = box_side / 2.0
    ix = (positions[:, 0] >= half).astype(int)
    iy = (positions[:, 1] >= half).astype(int)
    iz = (positions[:, 2] >= half).astype(int)
    return ix + 2 * iy + 4 * iz


def jackknife_delta_by_block(delta_values, observer_positions, box_side, n_blocks=8):
    """Leave-one-octant-out jackknife SE -- accounts for correlation between
    spatially nearby domains, unlike a naive per-domain standard error."""
    blocks = assign_octant(observer_positions, box_side)
    loo_means = []
    for b in range(n_blocks):
        keep = blocks != b
        if keep.sum() < 5:
            continue
        loo_means.append(delta_values[keep].mean())
    loo_means = np.array(loo_means)
    n = len(loo_means)
    full_mean = delta_values.mean()
    se_jack = np.sqrt((n - 1) / n * np.sum((loo_means - loo_means.mean()) ** 2))
    return full_mean, se_jack, loo_means


def assign_block_fine(positions, box_side, n_per_axis=3):
    idx = np.floor(positions / box_side * n_per_axis).astype(int)
    idx = np.clip(idx, 0, n_per_axis - 1)
    return idx[:, 0] + n_per_axis * idx[:, 1] + n_per_axis**2 * idx[:, 2]


def jackknife_delta_by_block_fine(delta_values, observer_positions, box_side, n_per_axis=3):
    """Finer block jackknife -- localises whether variance is driven by a
    small number of specific regions or diffusely spread."""
    blocks = assign_block_fine(observer_positions, box_side, n_per_axis)
    n_blocks = n_per_axis**3
    loo_means, block_ids, block_sizes = [], [], []
    for b in range(n_blocks):
        keep = blocks != b
        n_excluded = int((~keep).sum())
        if keep.sum() < 5 or n_excluded == 0:
            continue
        loo_means.append(delta_values[keep].mean())
        block_ids.append(b)
        block_sizes.append(n_excluded)
    loo_means = np.array(loo_means)
    n = len(loo_means)
    full_mean = delta_values.mean()
    se_jack = np.sqrt((n - 1) / n * np.sum((loo_means - loo_means.mean()) ** 2))
    return full_mean, se_jack, loo_means, block_ids, block_sizes


## 2. Mass-bin field construction

5 bins: fixed, previously-validated low- and high-mass edges; the transition zone between them split into N_TRANSITION_BINS equipopulated bins.

In [ ]:
def build_fields_refined_bins(tracers, velocities, masses, mass_lo_edge, mass_hi_edge, n_transition_bins):
    edges_all = [masses.min(), mass_lo_edge]
    mid_mask = (masses >= mass_lo_edge) & (masses < mass_hi_edge)
    mid_edges = np.quantile(masses[mid_mask], np.linspace(0, 1, n_transition_bins + 1))
    edges_all += list(mid_edges[1:])
    edges_all += [masses.max()]
    edges_all = np.array(edges_all)

    out = []
    for i in range(len(edges_all) - 1):
        lo, hi = edges_all[i], edges_all[i + 1]
        sel = (masses >= lo) & (masses <= hi if i == len(edges_all) - 2 else masses < hi)
        if sel.sum() < 500:
            out.append({"theta": None, "sigma2": None, "N": int(sel.sum()), "mass_median": np.nan})
            continue
        vx, vy, vz = build_velocity_field(tracers[sel], velocities[sel], GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
        theta, sigma2 = compute_theta_sigma2(vx.astype(np.float64), vy.astype(np.float64), vz.astype(np.float64), CELL_SIZE)
        out.append({"theta": theta, "sigma2": sigma2, "N": int(sel.sum()), "mass_median": float(np.median(masses[sel]))})
    return edges_all, out


## 3. Download all variants, N-match, build fields per mass bin

Fiducial plus three comparison variants (NoCooling, Jet, fgas-8sigma) at z=0.0. This is the expensive step (5 bins x 4 variants = 20 field builds, ~15-20 minutes); everything after this reuses these fields and is fast.

In [ ]:
print("Downloading fiducial, z=0.0...")
tr_f, vel_f, mass_f, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_A["run_path"], SNAPSHOT_Z0, expected_z=0.0)
print(f"  {len(tr_f)} tracers")

variant_data = {"fiducial": (tr_f, vel_f, mass_f)}
for var in COMPARISON_VARIANTS:
    print(f"Downloading {var['label']}, z=0.0...")
    tr_v, vel_v, mass_v, _ = load_tracers_velocities_masses(flamingo_dir, var["run_path"], SNAPSHOT_Z0, expected_z=0.0)
    print(f"  {len(tr_v)} tracers")
    variant_data[var["label"]] = (tr_v, vel_v, mass_v)

target_n = min(len(v[0]) for v in variant_data.values())
print(f"\nN-matching to {target_n} tracers across all variants")

edges_refined = None
fields_by_variant_refined = {}
t0 = time.time()
for label, (tr_v, vel_v, mass_v) in variant_data.items():
    tr_m, vel_m, mass_m = subsample_to_match_with_mass(tr_v, vel_v, mass_v, target_n, seed=SEED)
    edges_refined, bins_out = build_fields_refined_bins(tr_m, vel_m, mass_m, MASS_LO_EDGE, MASS_HI_EDGE, N_TRANSITION_BINS)
    fields_by_variant_refined[label] = bins_out
    print(f"  {label}: {[b['N'] for b in bins_out]} tracers per bin  ({time.time()-t0:.0f}s elapsed)")

print(f"\nFinal bin edges (Msun): {[f'{e:.2e}' for e in edges_refined]}")


## 4. Domain aggregation at two radii, mass-bin profile

R=15 Mpc (N=800) and R=30 Mpc (N=500). All three comparison variants tested across all 5 mass bins.

In [ ]:
DOMAIN_CONFIGS = [
    {"radius": 15.0, "n_obs": 800, "label": "R=15 Mpc"},
    {"radius": 30.0, "n_obs": 500, "label": "R=30 Mpc"},
]

results_refined = {}
for cfg in DOMAIN_CONFIGS:
    R, N_OBS, label_cfg = cfg["radius"], cfg["n_obs"], cfg["label"]
    obs_pos = make_observer_positions(N_OBS, 2 * R, BOX_SIDE, seed=SEED)
    print(f"\n{'='*70}\n{label_cfg}: {len(obs_pos)} domains\n{'='*70}")
    results_refined[label_cfg] = {}
    for var in COMPARISON_VARIANTS:
        bin_results = []
        for i in range(len(fields_by_variant_refined["fiducial"])):
            f_fid = fields_by_variant_refined["fiducial"][i]
            f_var = fields_by_variant_refined[var["label"]][i]
            if f_fid["theta"] is None or f_var["theta"] is None:
                bin_results.append(None)
                continue
            res = paired_QD_comparison(f_fid["theta"], f_fid["sigma2"], f_var["theta"], f_var["sigma2"],
                                        grid_tree, obs_pos, R, "fiducial", var["label"], verbose=False)
            bin_results.append({"mass_median": f_fid["mass_median"], "delta": res["delta"].mean(),
                                 "ci_lo": res["ci_lo"], "ci_hi": res["ci_hi"], "p": res["p_wilcoxon"]})
        results_refined[label_cfg][var["label"]] = bin_results
        print(f"\n  {var['label']}:")
        for i, b in enumerate(bin_results):
            if b is None:
                print(f"    Bin {i}: insufficient data"); continue
            print(f"    Bin {i} (median mass {b['mass_median']:.2e}): "
                  f"Delta={b['delta']:+.4f}  CI95%=[{b['ci_lo']:+.3f},{b['ci_hi']:+.3f}]  p={b['p']:.2e}")


## 5. Specificity screening: multiple-comparisons correction

Jet and fgas-8sigma are expected-null controls. With 20 (variant x radius x bin) comparisons, a naive p<0.05 count will show more nominal hits than genuinely significant ones. Benjamini-Hochberg FDR correction identifies which (if any) survive.

In [ ]:
def benjamini_hochberg(tests, alpha=0.05):
    """tests: list of (name, p). Returns sorted list with a 'survives' flag."""
    sorted_tests = sorted(tests, key=lambda t: t[1])
    m = len(sorted_tests)
    max_i = 0
    for i, t in enumerate(sorted_tests, start=1):
        if t[1] <= (i / m) * alpha:
            max_i = i
    return [(name, pval, i <= max_i) for i, (name, pval) in enumerate(sorted_tests, start=1)]


null_variant_tests = []
for label_cfg in results_refined:
    for var_label in ["Jet", "fgas-8sigma"]:
        for i, b in enumerate(results_refined[label_cfg][var_label]):
            if b is not None:
                null_variant_tests.append((f"{var_label} {label_cfg} bin{i}", b["p"]))

print(f"Total null-variant tests: {len(null_variant_tests)}")
n_nominal = sum(1 for _, p in null_variant_tests if p < 0.05)
print(f"Nominally significant (p<0.05), uncorrected: {n_nominal} (expected by chance: ~{len(null_variant_tests)*0.05:.1f})\n")

bh_results = benjamini_hochberg(null_variant_tests)
print("Benjamini-Hochberg correction (alpha=0.05):")
for name, pval, survives in bh_results:
    marker = "  <-- SURVIVES CORRECTION" if survives else ""
    print(f"  {name:30s} p={pval:.5f}{marker}")


## 6. Four-level robustness protocol

Applied to any signal that survives the specificity screen. Stages (I)-(II) use the original domain count (N=800 at R=15, N=500 at R=30); stage (III) increases the domain count (N=3000 / N=2000) to resolve borderline cases; stage (IV) is a deterministic cross-check independent of both domain count and random seed. The function returns every intermediate value needed for the summary plots in Section 8, so nothing is recomputed there.

In [ ]:
def run_robustness_protocol(bin_idx, variant_label, R, N_obs_initial, N_obs_replication,
                             seeds, n_grid_axis, min_separation_factor=2):
    f_fid = fields_by_variant_refined["fiducial"][bin_idx]
    f_var = fields_by_variant_refined[variant_label][bin_idx]

    print(f"\n--- {variant_label}, bin {bin_idx} (median mass {f_fid['mass_median']:.2e}), R={R:.0f} Mpc ---")

    # Stage i + ii: standard test + octant jackknife, original N, original seed
    obs_pos0 = make_observer_positions(N_obs_initial, min_separation_factor * R, BOX_SIDE, seed=seeds[0])
    res0 = paired_QD_comparison(f_fid["theta"], f_fid["sigma2"], f_var["theta"], f_var["sigma2"],
                                 grid_tree, obs_pos0, R, "fiducial", variant_label, verbose=False)
    delta0 = res0["delta"]
    se_naive0 = delta0.std(ddof=1) / np.sqrt(len(delta0))
    z_naive0 = delta0.mean() / se_naive0 if se_naive0 > 0 else float("nan")
    mean_j, se_j, loo = jackknife_delta_by_block(delta0, obs_pos0, BOX_SIDE, n_blocks=8)
    z_j = mean_j / se_j if se_j > 0 else float("nan")
    print(f"Stage i (standard, N={N_obs_initial}): Delta={delta0.mean():+.4f}  p={res0['p_wilcoxon']:.2e}  z_naive={z_naive0:+.2f}")
    print(f"Stage ii (octant jackknife, N={N_obs_initial}): z={z_j:+.2f}")

    if abs(z_j) < 2.5:
        mean_f, se_f, loo_f, ids_f, sizes_f = jackknife_delta_by_block_fine(delta0, obs_pos0, BOX_SIDE, n_per_axis=3)
        print(f"  Finer 27-block jackknife: SE={se_f:.4f} (vs 8-octant SE={se_j:.4f})")

    # Stage iii: multi-seed replication at increased domain count
    print(f"Stage iii (replication, N={N_obs_replication}, {len(seeds)} seeds):")
    z_jackknife_replication = []
    p_replication = []
    for s in seeds:
        obs_pos_s = make_observer_positions(N_obs_replication, min_separation_factor * R, BOX_SIDE, seed=s)
        res_s = paired_QD_comparison(f_fid["theta"], f_fid["sigma2"], f_var["theta"], f_var["sigma2"],
                                      grid_tree, obs_pos_s, R, "fiducial", variant_label, verbose=False)
        mean_js, se_js, _ = jackknife_delta_by_block(res_s["delta"], obs_pos_s, BOX_SIDE, n_blocks=8)
        z_js = mean_js / se_js if se_js > 0 else float("nan")
        z_jackknife_replication.append(z_js)
        p_replication.append(res_s["p_wilcoxon"])
        print(f"  seed {s:4d}: Delta={res_s['delta'].mean():+.4f}  p={res_s['p_wilcoxon']:.2e}  z_jackknife={z_js:+.2f}")

    # Stage iv: fixed grid cross-check (deterministic, no seed)
    obs_pos_grid = make_grid_observer_positions(n_grid_axis, BOX_SIDE)
    spacing = BOX_SIDE / n_grid_axis
    res_g = paired_QD_comparison(f_fid["theta"], f_fid["sigma2"], f_var["theta"], f_var["sigma2"],
                                  grid_tree, obs_pos_grid, R, "fiducial", variant_label, verbose=False)
    mean_jg, se_jg, _ = jackknife_delta_by_block(res_g["delta"], obs_pos_grid, BOX_SIDE, n_blocks=8)
    z_jg = mean_jg / se_jg if se_jg > 0 else float("nan")
    print(f"Stage iv (fixed grid, {len(obs_pos_grid)} points, spacing {spacing:.1f} Mpc): "
          f"Delta={res_g['delta'].mean():+.4f}  p={res_g['p_wilcoxon']:.2e}  z_jackknife={z_jg:+.2f}")

    return {
        "stage_i": res0,
        "z_naive_initial": z_naive0,
        "z_jackknife_initial": z_j,
        "z_jackknife_replication": z_jackknife_replication,
        "p_replication": p_replication,
        "stage_iv_grid": res_g,
        "z_jackknife_grid": z_jg,
    }


REPLICATION_SEEDS = [42, 123, 456, 789]

print("="*70, "\nLOW-MASS PILLAR (bin 0)\n", "="*70, sep="")
result_lowmass_R15 = run_robustness_protocol(0, "NoCooling", 15.0, 800, 3000, REPLICATION_SEEDS, n_grid_axis=16)
result_lowmass_R30 = run_robustness_protocol(0, "NoCooling", 30.0, 500, 2000, REPLICATION_SEEDS, n_grid_axis=12)

print("\n" + "="*70 + "\nHIGH-MASS PILLAR (bin 4)\n" + "="*70)
print("(shown for completeness/symmetry -- was already conclusive at stage i-ii in practice)")
result_highmass_R15 = run_robustness_protocol(4, "NoCooling", 15.0, 800, 800, REPLICATION_SEEDS[:1], n_grid_axis=16)
result_highmass_R30 = run_robustness_protocol(4, "NoCooling", 30.0, 500, 500, REPLICATION_SEEDS[:1], n_grid_axis=12)


## 7. Diagnosing the sole surviving null-variant test

If Section 5 flags a single (variant, bin, radius) combination surviving BH-FDR correction, apply the same protocol to it directly -- a lone survivor among many comparisons is exactly the case a replication check is designed to resolve.

In [ ]:
# identify which mass bin index corresponds to the flagged combination
# (update these indices/labels if Section 5's output flags a different one)
FLAGGED_VARIANT = "Jet"
FLAGGED_BIN = 3
FLAGGED_R = 15.0
FLAGGED_N = 800

print(f"Diagnosing: {FLAGGED_VARIANT}, bin {FLAGGED_BIN}, R={FLAGGED_R:.0f} Mpc\n")
result_anomaly = run_robustness_protocol(FLAGGED_BIN, FLAGGED_VARIANT, FLAGGED_R, FLAGGED_N, FLAGGED_N,
                                          REPLICATION_SEEDS, n_grid_axis=16)


## 8. Summary plots

Panel 1: mass-bin profile. Panel 2: seed-to-seed stability (low-mass pillar vs the diagnosed anomaly), reusing the results already computed in Sections 6-7 -- nothing is recomputed. Panel 3 is deliberately split into two sub-plots that never share an axis implying a single continuous 'evolution': the left isolates the effect of method alone at fixed N; the right isolates the effect of statistical power alone at fixed method, with the fixed-grid cross-check shown as a distinctly shaped marker, explicitly not on the same N/scheme as the points beside it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: mass-bin profile at both radii
ax = axes[0]
colors = {"NoCooling": "tab:blue", "Jet": "tab:green", "fgas-8sigma": "tab:red"}
linestyles = {"R=15 Mpc": "-", "R=30 Mpc": "--"}
for label_cfg, ls in linestyles.items():
    for var_label, bin_results in results_refined[label_cfg].items():
        valid = [b for b in bin_results if b is not None]
        masses = [b["mass_median"] for b in valid]
        deltas = [b["delta"] for b in valid]
        lo = [b["delta"] - b["ci_lo"] for b in valid]
        hi = [b["ci_hi"] - b["delta"] for b in valid]
        ax.errorbar(masses, deltas, yerr=[lo, hi], fmt="o" + ls, capsize=3,
                     color=colors.get(var_label), alpha=0.9 if ls == "-" else 0.5,
                     label=f"{var_label} ({label_cfg})")
ax.axhline(0, color="grey", linestyle=":", linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("Halo mass [M$_\\odot$]")
ax.set_ylabel("$\\Delta Q_D$ = variant $-$ fiducial")
ax.set_title("Mass-bin profile, z=0.0")
ax.legend(fontsize=7)

# Panel 2: seed stability, low-mass pillar vs the diagnosed anomaly --
# reuses p_replication already stored in Sections 6-7, no recomputation
ax = axes[1]
seeds_x = list(range(len(REPLICATION_SEEDS)))
logp_lowmass = [-np.log10(p) for p in result_lowmass_R15["p_replication"]]
logp_anomaly = [-np.log10(p) for p in result_anomaly["p_replication"]]
ax.plot(seeds_x, logp_lowmass, "o-", color="tab:orange",
        label=f"Low mass, R=15 (N={3000})")
ax.plot(seeds_x, logp_anomaly, "o-", color="tab:green",
        label=f"{FLAGGED_VARIANT} bin{FLAGGED_BIN}, R={FLAGGED_R:.0f} (N={FLAGGED_N})")
ax.axhline(-np.log10(0.05), color="grey", linestyle="--", linewidth=1, label="p=0.05")
ax.set_xticks(seeds_x)
ax.set_xticklabels([f"seed {s}" for s in REPLICATION_SEEDS])
ax.set_ylabel("$-\\log_{10}(p)$ (Wilcoxon)")
ax.set_title("Stability across seeds\n(different N per series -- see labels)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("mass_signal_summary_1.png", dpi=150)
plt.show()


In [ ]:
# Second figure: the robustness path of the low-mass pillar, split into two
# panels that never mix "what changed" on a single implied timeline --
# built entirely from the dictionaries already returned in Section 6.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

naive_z = {"R=15 Mpc": result_lowmass_R15["z_naive_initial"], "R=30 Mpc": result_lowmass_R30["z_naive_initial"]}
jack_z_lowN = {"R=15 Mpc": result_lowmass_R15["z_jackknife_initial"], "R=30 Mpc": result_lowmass_R30["z_jackknife_initial"]}
jack_z_highN_range = {
    "R=15 Mpc": (min(result_lowmass_R15["z_jackknife_replication"]), max(result_lowmass_R15["z_jackknife_replication"])),
    "R=30 Mpc": (min(result_lowmass_R30["z_jackknife_replication"]), max(result_lowmass_R30["z_jackknife_replication"])),
}
grid_z = {"R=15 Mpc": result_lowmass_R15["z_jackknife_grid"], "R=30 Mpc": result_lowmass_R30["z_jackknife_grid"]}

ax = axes[0]
x = np.arange(2)
width = 0.35
r15_vals = [naive_z["R=15 Mpc"], jack_z_lowN["R=15 Mpc"]]
r30_vals = [naive_z["R=30 Mpc"], jack_z_lowN["R=30 Mpc"]]
ax.bar(x - width/2, r15_vals, width, label="R=15 Mpc (N=800)", color="tab:orange")
ax.bar(x + width/2, r30_vals, width, label="R=30 Mpc (N=500)", color="tab:red")
ax.axhline(2, color="grey", linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(["Naive", "Jackknife"])
ax.set_ylabel("z")
ax.set_title("Effect of method alone, N fixed")
ax.legend()

ax = axes[1]
for label, color, offset in [("R=15 Mpc", "tab:orange", -0.15), ("R=30 Mpc", "tab:red", 0.15)]:
    lo, hi = jack_z_highN_range[label]
    mid = (lo + hi) / 2
    ax.plot([1 + offset, 1 + offset], [lo, hi], "-", color=color, linewidth=2)
    ax.plot(1 + offset, mid, "o", color=color)
    ax.plot(0 + offset, jack_z_lowN[label], "o", color=color, alpha=0.5)
    ax.plot(2 + offset, grid_z[label], "^", color=color, markersize=9)
ax.axhline(2, color="grey", linestyle="--", linewidth=1)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Jackknife\nN=800/500\n(single seed)", "Jackknife\nN=3000/2000\n(4-seed range)",
                     "Fixed grid\n(different N & scheme --\nindependent check)"])
ax.set_ylabel("z (jackknife)")
ax.set_title("Effect of power alone, method fixed\n+ independent check")
ax.plot([], [], "o", color="grey", label="Single point")
ax.plot([], [], "^", color="grey", label="Grid (independent check)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("mass_signal_summary_2.png", dpi=150)
plt.show()


## 9. Follow-up: gas thermodynamic properties (Section 6.1 of the companion report)

Independent, purely thermodynamic check of the transition-mass hypothesis, using SOAP's GasMass, HotGasMass, and GasTemperature fields (same SO/200_crit aperture used throughout this notebook). Three sub-checks: mean vs median robustness, specificity against Jet/fgas-8sigma, and evolution between z=0.0 and z=1.0.

In [ ]:
def load_gas_properties(flamingo_dir, run_path, snapshot_file, expected_z=None):
    """GasMass, HotGasMass, GasTemperature under SO/200_crit -- same aperture
    used throughout this notebook, same halo selection (M > MASS_CUT)."""
    soap_file = flamingo_dir[f"{run_path}/SOAP-HBT/{snapshot_file}"]
    z = float(np.array(soap_file["Header"].attrs["Redshift"]).squeeze())
    if expected_z is not None:
        assert abs(z - expected_z) < 1e-6

    total_mass_raw = np.array(soap_file["SO/200_crit/TotalMass"][:], dtype=np.float64)
    conv_mass = dict(soap_file["SO/200_crit/TotalMass"].attrs)
    cgs_mass = float(np.array(conv_mass["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    mass_msun = total_mass_raw * cgs_mass / GRAMS_PER_MSUN
    mask = mass_msun > MASS_CUT

    gas_mass_raw = np.array(soap_file["SO/200_crit/GasMass"][:], dtype=np.float64)
    conv_gas = dict(soap_file["SO/200_crit/GasMass"].attrs)
    cgs_gas = float(np.array(conv_gas["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    gas_mass_msun = gas_mass_raw * cgs_gas / GRAMS_PER_MSUN

    hot_gas_mass_raw = np.array(soap_file["SO/200_crit/HotGasMass"][:], dtype=np.float64)
    conv_hot = dict(soap_file["SO/200_crit/HotGasMass"].attrs)
    cgs_hot = float(np.array(conv_hot["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    hot_gas_mass_msun = hot_gas_mass_raw * cgs_hot / GRAMS_PER_MSUN

    gas_temp_raw = np.array(soap_file["SO/200_crit/GasTemperature"][:], dtype=np.float64)
    conv_temp = dict(soap_file["SO/200_crit/GasTemperature"].attrs)
    cgs_temp = float(np.array(conv_temp["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    gas_temp_K = gas_temp_raw * cgs_temp

    return mass_msun[mask], gas_mass_msun[mask], hot_gas_mass_msun[mask], gas_temp_K[mask], z


SNAPSHOT_Z1 = "halo_properties_0057.hdf5"  # z=1.0, already used elsewhere in this project

# fixed, log-spaced mass edges, IDENTICAL at both epochs -- any shift in the
# crossing point is then attributable to physics, not to different binning
N_FINE_BINS = 12
mass_edges_fine = np.logspace(np.log10(MASS_CUT), np.log10(2e13), N_FINE_BINS + 1)


def gas_temp_profile(mass_arr, temp_arr, edges):
    medians, means, medians_mass, counts = [], [], [], []
    for i in range(len(edges) - 1):
        sel = (mass_arr >= edges[i]) & (mass_arr < edges[i + 1])
        n = int(sel.sum())
        counts.append(n)
        if n < 50:
            medians.append(np.nan); means.append(np.nan); medians_mass.append(np.nan)
            continue
        medians.append(np.nanmedian(temp_arr[sel]))
        means.append(np.nanmean(temp_arr[sel]))
        medians_mass.append(np.median(mass_arr[sel]))
    return np.array(medians_mass), np.array(medians), np.array(means), np.array(counts)


def print_ratio_table(mass_ref, temp_ref, mass_var, temp_var, edges, label):
    m_ref, med_ref, mean_ref, n_ref = gas_temp_profile(mass_ref, temp_ref, edges)
    m_var, med_var, mean_var, n_var = gas_temp_profile(mass_var, temp_var, edges)
    print(f"\n{label}")
    print(f"{'mass':>12s} {'N ref.':>7s} {'N var.':>7s} {'median ratio':>13s} {'mean ratio':>11s}")
    for i in range(len(edges) - 1):
        if n_ref[i] < 50 or n_var[i] < 50:
            continue
        rap_med = med_var[i] / med_ref[i]
        rap_mean = mean_var[i] / mean_ref[i]
        print(f"{m_ref[i]:12.2e} {n_ref[i]:7d} {n_var[i]:7d} {rap_med:13.3f} {rap_mean:11.3f}")
    return m_ref, med_ref, med_var


In [ ]:
print("Downloading gas properties, z=0.0, Jet and fgas-8sigma...")
mass_jet_z0, gasmass_jet_z0, hotgas_jet_z0, temp_jet_z0, _ = load_gas_properties(
    flamingo_dir, VARIANT_JET["run_path"], SNAPSHOT_Z0, expected_z=0.0)
mass_fgas_z0, gasmass_fgas_z0, hotgas_fgas_z0, temp_fgas_z0, _ = load_gas_properties(
    flamingo_dir, VARIANT_FGAS8SIGMA["run_path"], SNAPSHOT_Z0, expected_z=0.0)

print("Downloading gas properties, fiducial and NoCooling, z=0.0 (for the ratio baseline)...")
mass_f, gasmass_f, hotgas_f, temp_f, _ = load_gas_properties(
    flamingo_dir, VARIANT_A["run_path"], SNAPSHOT_Z0, expected_z=0.0)
mass_n, gasmass_n, hotgas_n, temp_n, _ = load_gas_properties(
    flamingo_dir, VARIANT_B["run_path"], SNAPSHOT_Z0, expected_z=0.0)

print("Downloading gas properties, z=1.0, all variants...")
mass_f_z1, gasmass_f_z1, hotgas_f_z1, temp_f_z1, _ = load_gas_properties(
    flamingo_dir, VARIANT_A["run_path"], SNAPSHOT_Z1, expected_z=1.0)
mass_n_z1, gasmass_n_z1, hotgas_n_z1, temp_n_z1, _ = load_gas_properties(
    flamingo_dir, VARIANT_B["run_path"], SNAPSHOT_Z1, expected_z=1.0)
mass_jet_z1, gasmass_jet_z1, hotgas_jet_z1, temp_jet_z1, _ = load_gas_properties(
    flamingo_dir, VARIANT_JET["run_path"], SNAPSHOT_Z1, expected_z=1.0)
mass_fgas_z1, gasmass_fgas_z1, hotgas_fgas_z1, temp_fgas_z1, _ = load_gas_properties(
    flamingo_dir, VARIANT_FGAS8SIGMA["run_path"], SNAPSHOT_Z1, expected_z=1.0)

# sanity check: cold gas fraction under SOAP's own hot/cold split -- expected
# to collapse to ~0 in NoCooling at every mass (see companion report, Section
# 6.1, for why this is a non-discriminating definitional artefact rather
# than a genuine null result)
valid_f = gasmass_f > 0
cold_frac_f = np.full_like(gasmass_f, np.nan)
cold_frac_f[valid_f] = (gasmass_f[valid_f] - hotgas_f[valid_f]) / gasmass_f[valid_f]
valid_n = gasmass_n > 0
cold_frac_n = np.full_like(gasmass_n, np.nan)
cold_frac_n[valid_n] = (gasmass_n[valid_n] - hotgas_n[valid_n]) / gasmass_n[valid_n]
print(f"\nSOAP cold gas fraction sanity check: median(fiducial)={np.nanmedian(cold_frac_f):.3f}  "
      f"median(NoCooling)={np.nanmedian(cold_frac_n):.3f} (expected ~0, non-discriminating by design)")

results_ratio = {}
results_ratio["z=0.0, NoCooling"] = print_ratio_table(mass_f, temp_f, mass_n, temp_n, mass_edges_fine, "z=0.0, NoCooling / fiducial")
results_ratio["z=0.0, Jet"] = print_ratio_table(mass_f, temp_f, mass_jet_z0, temp_jet_z0, mass_edges_fine, "z=0.0, Jet / fiducial")
results_ratio["z=0.0, fgas-8sigma"] = print_ratio_table(mass_f, temp_f, mass_fgas_z0, temp_fgas_z0, mass_edges_fine, "z=0.0, fgas-8sigma / fiducial")
results_ratio["z=1.0, NoCooling"] = print_ratio_table(mass_f_z1, temp_f_z1, mass_n_z1, temp_n_z1, mass_edges_fine, "z=1.0, NoCooling / fiducial")
results_ratio["z=1.0, Jet"] = print_ratio_table(mass_f_z1, temp_f_z1, mass_jet_z1, temp_jet_z1, mass_edges_fine, "z=1.0, Jet / fiducial")
results_ratio["z=1.0, fgas-8sigma"] = print_ratio_table(mass_f_z1, temp_f_z1, mass_fgas_z1, temp_fgas_z1, mass_edges_fine, "z=1.0, fgas-8sigma / fiducial")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_gas = {"NoCooling": "tab:blue", "Jet": "tab:green", "fgas-8sigma": "tab:red"}

for ax, epoch_label in zip(axes, ["z=0.0", "z=1.0"]):
    for var_label in ["NoCooling", "Jet", "fgas-8sigma"]:
        key = f"{epoch_label}, {var_label}"
        m_ref, med_ref, med_var = results_ratio[key]
        valid = ~np.isnan(med_ref) & ~np.isnan(med_var)
        ratio = med_var[valid] / med_ref[valid]
        ax.plot(m_ref[valid], ratio, "o-", color=colors_gas[var_label], label=var_label)
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xlabel("Halo mass [M$_\\odot$]")
    ax.set_ylabel("T$_{gas}$(variant) / T$_{gas}$(fiducial)  [median]")
    ax.set_title(f"Gas temperature ratio, {epoch_label}")
    ax.legend()

plt.tight_layout()
plt.savefig("gas_temperature_ratio_summary.png", dpi=150)
plt.show()


## 10. Follow-up: velocity-field anisotropy (Section 6.2 of the companion report)

Cosmic-web classification (velocity shear tensor eigenvalues, Hoffman et al. 2012) applied to haloes in each mass bin, testing whether the fiducial run shows more anisotropic/filamentary infall than NoCooling. Two methodological corrections were required and are documented in the function below: (1) the trace-free shear tensor used for Q_D's sigma2 makes the Knot class mathematically impossible at any positive threshold (three eigenvalues of a trace-free tensor cannot all be positive); (2) the sign convention (positive = converging, Hoffman et al. 2012) must be applied explicitly. Both are fixed in `compute_vweb_eigenvalues_grid` below, which uses the FULL (non-trace-free) symmetric velocity-gradient tensor, sign-flipped.

In [ ]:
def compute_vweb_eigenvalues_grid(vx, vy, vz, cell_size):
    """Eigenvalues of the FULL (non-trace-free) symmetric velocity-gradient
    tensor, sign-flipped so that positive=converging (Hoffman et al. 2012
    convention). NOT the same tensor as compute_theta_sigma2's sigma (which
    is trace-free and therefore cannot yield a Knot classification at any
    positive threshold -- verified analytically and confirmed empirically
    during this analysis)."""
    dvx_dx = (np.roll(vx,-1,0)-np.roll(vx,1,0))/(2*cell_size)
    dvx_dy = (np.roll(vx,-1,1)-np.roll(vx,1,1))/(2*cell_size)
    dvx_dz = (np.roll(vx,-1,2)-np.roll(vx,1,2))/(2*cell_size)
    dvy_dx = (np.roll(vy,-1,0)-np.roll(vy,1,0))/(2*cell_size)
    dvy_dy = (np.roll(vy,-1,1)-np.roll(vy,1,1))/(2*cell_size)
    dvy_dz = (np.roll(vy,-1,2)-np.roll(vy,1,2))/(2*cell_size)
    dvz_dx = (np.roll(vz,-1,0)-np.roll(vz,1,0))/(2*cell_size)
    dvz_dy = (np.roll(vz,-1,1)-np.roll(vz,1,1))/(2*cell_size)
    dvz_dz = (np.roll(vz,-1,2)-np.roll(vz,1,2))/(2*cell_size)
    theta = dvx_dx+dvy_dy+dvz_dz
    grad = np.array([[dvx_dx,dvx_dy,dvx_dz],[dvy_dx,dvy_dy,dvy_dz],[dvz_dx,dvz_dy,dvz_dz]])
    sym = 0.5*(grad+grad.transpose(1,0,2,3,4))
    sym_conv = -sym
    n_grid = vx.shape[0]
    sym_batch = sym_conv.transpose(2,3,4,0,1).reshape(-1,3,3)
    eigvals = np.linalg.eigvalsh(sym_batch)
    eigvals = eigvals.reshape(n_grid, n_grid, n_grid, 3)
    return eigvals, theta


def lookup_grid_class_at_positions(classification, positions, box_side, n_grid):
    cell_size = box_side / n_grid
    idx = np.round(positions / cell_size).astype(int) % n_grid
    return classification[idx[:, 0], idx[:, 1], idx[:, 2]]


def compute_filknot_diff_by_bin(class_f, class_variant, tr_f_m, tr_variant_m,
                                 mass_f_m, mass_variant_m, edges_refined,
                                 box_side, n_grid, label, verbose=True):
    """Delta = variant - fiducial (sign convention consistent with the rest
    of this project)."""
    results = []
    if verbose:
        print(f"\n{label} vs fiducial:")
    for i in range(len(edges_refined) - 1):
        lo, hi = edges_refined[i], edges_refined[i + 1]
        sel_f = (mass_f_m >= lo) & (mass_f_m <= hi if i == len(edges_refined) - 2 else mass_f_m < hi)
        sel_v = (mass_variant_m >= lo) & (mass_variant_m <= hi if i == len(edges_refined) - 2 else mass_variant_m < hi)

        class_at_halos_f = lookup_grid_class_at_positions(class_f, tr_f_m[sel_f], box_side, n_grid)
        class_at_halos_v = lookup_grid_class_at_positions(class_variant, tr_variant_m[sel_v], box_side, n_grid)

        n_f = len(class_at_halos_f); n_v = len(class_at_halos_v)
        k_f = int(np.sum(class_at_halos_f >= 2)); k_v = int(np.sum(class_at_halos_v >= 2))
        p_f = k_f / n_f; p_v = k_v / n_v
        p_pool = (k_f + k_v) / (n_f + n_v)
        se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_f + 1 / n_v))
        z = (p_v - p_f) / se if se > 0 else float("nan")
        p_val = 2 * (1 - norm.cdf(abs(z)))
        mass_med = np.median(mass_f_m[sel_f])
        results.append({"mass_median": mass_med, "diff": p_v - p_f, "z": z, "p": p_val})
        if verbose:
            print(f"  Bin {i} (mass {mass_med:.2e}): Delta={p_v - p_f:+.4f}  z={z:+.2f}  p={p_val:.2e}")
    return results


### 10.1 z=0.0: download all 4 variants, build full-catalogue fields, calibrate threshold

In [ ]:
from scipy.stats import norm

VARIANT_JET = {"run_path": "L1_m9/Jet", "label": "Jet"}
VARIANT_FGAS8SIGMA = {"run_path": "L1_m9/fgas-8sigma", "label": "fgas-8sigma"}

print("Downloading Jet, z=0.0...")
tr_jet, vel_jet, mass_jet, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_JET["run_path"], SNAPSHOT_Z0, expected_z=0.0)
print("Downloading fgas-8sigma, z=0.0...")
tr_fgas, vel_fgas, mass_fgas, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_FGAS8SIGMA["run_path"], SNAPSHOT_Z0, expected_z=0.0)

tr_jet_m, vel_jet_m, mass_jet_m = subsample_to_match_with_mass(tr_jet, vel_jet, mass_jet, target_n, seed=SEED)
tr_fgas_m, vel_fgas_m, mass_fgas_m = subsample_to_match_with_mass(tr_fgas, vel_fgas, mass_fgas, target_n, seed=SEED)

print("Building full-catalogue velocity field, fiducial (z=0.0)...")
vx_f0, vy_f0, vz_f0 = build_velocity_field(tr_f_m, vel_f_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_f0, _ = compute_vweb_eigenvalues_grid(vx_f0.astype(np.float64), vy_f0.astype(np.float64), vz_f0.astype(np.float64), CELL_SIZE)

print("Building full-catalogue velocity field, NoCooling (z=0.0)...")
vx_n0, vy_n0, vz_n0 = build_velocity_field(tr_n_m, vel_n_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_n0, _ = compute_vweb_eigenvalues_grid(vx_n0.astype(np.float64), vy_n0.astype(np.float64), vz_n0.astype(np.float64), CELL_SIZE)

print("Building full-catalogue velocity field, Jet (z=0.0)...")
vx_jet0, vy_jet0, vz_jet0 = build_velocity_field(tr_jet_m, vel_jet_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_jet0, _ = compute_vweb_eigenvalues_grid(vx_jet0.astype(np.float64), vy_jet0.astype(np.float64), vz_jet0.astype(np.float64), CELL_SIZE)

print("Building full-catalogue velocity field, fgas-8sigma (z=0.0)...")
vx_fgas0, vy_fgas0, vz_fgas0 = build_velocity_field(tr_fgas_m, vel_fgas_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_fgas0, _ = compute_vweb_eigenvalues_grid(vx_fgas0.astype(np.float64), vy_fgas0.astype(np.float64), vz_fgas0.astype(np.float64), CELL_SIZE)

print("\nThreshold scan (fiducial, z=0.0) -- target Void~0.31/Sheet~0.48/Filament~0.19/Knot~0.015")
for th in [3.0, 4.0, 5.0, 5.5, 6.0, 6.5, 7.0]:
    cls = np.sum(eigvals_f0 > th, axis=-1)
    frac = np.bincount(cls.ravel(), minlength=4) / cls.size
    print(f"  th={th:.1f}: Void={frac[0]:.3f} Sheet={frac[1]:.3f} Filament={frac[2]:.3f} Knot={frac[3]:.3f}")


### 10.2 z=0.0: classify and compare (NoCooling, Jet, fgas-8sigma vs fiducial)

Calibrated threshold from 10.1 was 6.0 at z=0.0.

In [ ]:
VWEB_THRESHOLD_Z0 = 6.0
class_f0 = np.sum(eigvals_f0 > VWEB_THRESHOLD_Z0, axis=-1)
class_n0 = np.sum(eigvals_n0 > VWEB_THRESHOLD_Z0, axis=-1)
class_jet0 = np.sum(eigvals_jet0 > VWEB_THRESHOLD_Z0, axis=-1)
class_fgas0 = np.sum(eigvals_fgas0 > VWEB_THRESHOLD_Z0, axis=-1)

results_nc_z0 = compute_filknot_diff_by_bin(class_f0, class_n0, tr_f_m, tr_n_m, mass_f_m, mass_n_m,
                                             edges_refined, BOX_SIDE, N_GRID, "NoCooling (z=0.0)")
results_jet_z0 = compute_filknot_diff_by_bin(class_f0, class_jet0, tr_f_m, tr_jet_m, mass_f_m, mass_jet_m,
                                              edges_refined, BOX_SIDE, N_GRID, "Jet (z=0.0)")
results_fgas_z0 = compute_filknot_diff_by_bin(class_f0, class_fgas0, tr_f_m, tr_fgas_m, mass_f_m, mass_fgas_m,
                                               edges_refined, BOX_SIDE, N_GRID, "fgas-8sigma (z=0.0)")


def benjamini_hochberg(tests, alpha=0.05):
    sorted_tests = sorted(tests, key=lambda t: t[1])
    m = len(sorted_tests)
    max_i = 0
    for i, t in enumerate(sorted_tests, start=1):
        if t[1] <= (i / m) * alpha:
            max_i = i
    return [(name, pval, i <= max_i) for i, (name, pval) in enumerate(sorted_tests, start=1)]


tests_z0 = [(f"Jet bin{i}", r["p"]) for i, r in enumerate(results_jet_z0)] + \
           [(f"fgas bin{i}", r["p"]) for i, r in enumerate(results_fgas_z0)]
print(f"\nBenjamini-Hochberg correction (z=0.0, 10 tests):")
for name, p, survives in benjamini_hochberg(tests_z0):
    marker = "  <-- SURVIVES" if survives else ""
    print(f"  {name:12s} p={p:.5f}{marker}")


### 10.3 z=1.0: repeat the full analysis at a second epoch

z=1.0 uses a different N-matching target (own tracer counts at this epoch) and its own threshold calibration -- do not assume it matches z=0.0's values.

In [ ]:
SNAPSHOT_Z1 = "halo_properties_0057.hdf5"  # z=1.0, L1_m9 numbering

print("Downloading fiducial, z=1.0..."); tr_f1, vel_f1, mass_f1, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_A["run_path"], SNAPSHOT_Z1, expected_z=1.0)
print("Downloading NoCooling, z=1.0..."); tr_n1, vel_n1, mass_n1, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_B["run_path"], SNAPSHOT_Z1, expected_z=1.0)
print("Downloading Jet, z=1.0..."); tr_jet1, vel_jet1, mass_jet1, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_JET["run_path"], SNAPSHOT_Z1, expected_z=1.0)
print("Downloading fgas-8sigma, z=1.0..."); tr_fgas1, vel_fgas1, mass_fgas1, _ = load_tracers_velocities_masses(flamingo_dir, VARIANT_FGAS8SIGMA["run_path"], SNAPSHOT_Z1, expected_z=1.0)

target_n_z1 = min(len(tr_f1), len(tr_n1), len(tr_jet1), len(tr_fgas1))
print(f"N-matching (z=1.0) to {target_n_z1} tracers")

tr_f1_m, vel_f1_m, mass_f1_m = subsample_to_match_with_mass(tr_f1, vel_f1, mass_f1, target_n_z1, seed=SEED)
tr_n1_m, vel_n1_m, mass_n1_m = subsample_to_match_with_mass(tr_n1, vel_n1, mass_n1, target_n_z1, seed=SEED)
tr_jet1_m, vel_jet1_m, mass_jet1_m = subsample_to_match_with_mass(tr_jet1, vel_jet1, mass_jet1, target_n_z1, seed=SEED)
tr_fgas1_m, vel_fgas1_m, mass_fgas1_m = subsample_to_match_with_mass(tr_fgas1, vel_fgas1, mass_fgas1, target_n_z1, seed=SEED)

print("Building full-catalogue velocity fields, z=1.0 (4 variants)...")
vx_f1, vy_f1, vz_f1 = build_velocity_field(tr_f1_m, vel_f1_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_f1, _ = compute_vweb_eigenvalues_grid(vx_f1.astype(np.float64), vy_f1.astype(np.float64), vz_f1.astype(np.float64), CELL_SIZE)

vx_n1, vy_n1, vz_n1 = build_velocity_field(tr_n1_m, vel_n1_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_n1, _ = compute_vweb_eigenvalues_grid(vx_n1.astype(np.float64), vy_n1.astype(np.float64), vz_n1.astype(np.float64), CELL_SIZE)

vx_jet1, vy_jet1, vz_jet1 = build_velocity_field(tr_jet1_m, vel_jet1_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_jet1, _ = compute_vweb_eigenvalues_grid(vx_jet1.astype(np.float64), vy_jet1.astype(np.float64), vz_jet1.astype(np.float64), CELL_SIZE)

vx_fgas1, vy_fgas1, vz_fgas1 = build_velocity_field(tr_fgas1_m, vel_fgas1_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
eigvals_fgas1, _ = compute_vweb_eigenvalues_grid(vx_fgas1.astype(np.float64), vy_fgas1.astype(np.float64), vz_fgas1.astype(np.float64), CELL_SIZE)

print("\nThreshold scan (fiducial, z=1.0)")
for th in [3.0, 4.0, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0]:
    cls = np.sum(eigvals_f1 > th, axis=-1)
    frac = np.bincount(cls.ravel(), minlength=4) / cls.size
    print(f"  th={th:.1f}: Void={frac[0]:.3f} Sheet={frac[1]:.3f} Filament={frac[2]:.3f} Knot={frac[3]:.3f}")
print("(calibrated value used below: 6.5 -- verify against this scan if re-running)")


In [ ]:
VWEB_THRESHOLD_Z1 = 6.5
class_f1c = np.sum(eigvals_f1 > VWEB_THRESHOLD_Z1, axis=-1)
class_n1c = np.sum(eigvals_n1 > VWEB_THRESHOLD_Z1, axis=-1)
class_jet1c = np.sum(eigvals_jet1 > VWEB_THRESHOLD_Z1, axis=-1)
class_fgas1c = np.sum(eigvals_fgas1 > VWEB_THRESHOLD_Z1, axis=-1)

results_nc_z1 = compute_filknot_diff_by_bin(class_f1c, class_n1c, tr_f1_m, tr_n1_m, mass_f1_m, mass_n1_m,
                                             edges_refined, BOX_SIDE, N_GRID, "NoCooling (z=1.0)")
results_jet_z1 = compute_filknot_diff_by_bin(class_f1c, class_jet1c, tr_f1_m, tr_jet1_m, mass_f1_m, mass_jet1_m,
                                              edges_refined, BOX_SIDE, N_GRID, "Jet (z=1.0)")
results_fgas_z1 = compute_filknot_diff_by_bin(class_f1c, class_fgas1c, tr_f1_m, tr_fgas1_m, mass_f1_m, mass_fgas1_m,
                                               edges_refined, BOX_SIDE, N_GRID, "fgas-8sigma (z=1.0)")

tests_z1 = [(f"Jet bin{i}", r["p"]) for i, r in enumerate(results_jet_z1)] + \
           [(f"fgas bin{i}", r["p"]) for i, r in enumerate(results_fgas_z1)]
print(f"\nBenjamini-Hochberg correction (z=1.0, 10 tests):")
for name, p, survives in benjamini_hochberg(tests_z1):
    marker = "  <-- SURVIVES" if survives else ""
    print(f"  {name:12s} p={p:.5f}{marker}")


### 10.4 Summary plot: both epochs, all three comparison variants

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
colors_vweb = {"NoCooling": "tab:blue", "Jet": "tab:green", "fgas-8sigma": "tab:red"}

for ax, epoch_label, results_nc_e, results_jet_e, results_fgas_e in [
    (axes[0], "z=0.0", results_nc_z0, results_jet_z0, results_fgas_z0),
    (axes[1], "z=1.0", results_nc_z1, results_jet_z1, results_fgas_z1),
]:
    for label, results in [("NoCooling", results_nc_e), ("Jet", results_jet_e), ("fgas-8sigma", results_fgas_e)]:
        masses = [r["mass_median"] for r in results]
        diffs = [r["diff"] for r in results]
        ax.plot(masses, diffs, "o-", color=colors_vweb[label], label=label)
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xlabel("Halo mass [M$_\\odot$]")
    ax.set_title(f"V-web anisotropy specificity, {epoch_label}")
    ax.legend()
axes[0].set_ylabel("$\\Delta$(Filament+Knot fraction) = variant - fiducial")

plt.tight_layout()
plt.savefig("vweb_anisotropy_both_epochs.png", dpi=150)
plt.show()
